# Pipeline completo no Colab: IMDb + recomendação de filmes

Roda, do zero, a integração das bases, os Modelos A e B e os gráficos do artigo.
Execute as células **na ordem** (Ambiente de execução → Executar tudo funciona, mas a célula 2 pede um arquivo).

Tempo estimado: 10 a 25 minutos (a leitura de `title_principals` é a parte lenta).

## 1. Pastas do projeto

In [ ]:
from pathlib import Path
RAIZ = Path("/content/projeto")
for pasta in ["dados_brutos", "dados", "resultados"]:
    (RAIZ / pasta).mkdir(parents=True, exist_ok=True)
print("Pastas criadas em", RAIZ)

## 2. Baixar os dados do Kaggle

**Opção A (recomendada): API do Kaggle.** No Kaggle: foto do perfil → **Settings** → seção **API** → **Create New Token**
(baixa um `kaggle.json`). Rode a célula abaixo e envie esse arquivo quando solicitado.
Antes, abra a página do dataset `kunwarakash/imdbdatasets` pelo menos uma vez, logado.

**Opção B:** se já baixou o ZIP do dataset, coloque-o no Google Drive e use a célula 2B.

In [ ]:
# Opção A - API do Kaggle
from google.colab import files
import os, shutil
print("Selecione o arquivo kaggle.json")
up = files.upload()
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move(next(iter(up)), "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
!kaggle datasets download -d kunwarakash/imdbdatasets -p /content/projeto/dados_brutos

In [ ]:
# Opção B - ZIP no Google Drive (ignore esta célula se usou a opção A)
# from google.colab import drive
# drive.mount("/content/drive")
# !cp "/content/drive/MyDrive/CAMINHO/imdbdatasets.zip" /content/projeto/dados_brutos/

## 3. Descompactar (inclusive ZIPs dentro de ZIPs) e listar os arquivos

In [ ]:
import zipfile
from pathlib import Path

pasta = Path("/content/projeto/dados_brutos")
while True:
    zips = list(pasta.rglob("*.zip"))
    if not zips:
        break
    for z in zips:
        with zipfile.ZipFile(z) as arq:
            arq.extractall(z.parent)
        z.unlink()

for p in sorted(pasta.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size/1e6:10.1f} MB  {p.relative_to(pasta)}")

## 4. Etapa 1 - Seleção, limpeza e integração
Compare a conferência impressa no final com o artigo.

In [ ]:
"""
Etapa 1 - Seleção, limpeza e integração das cinco tabelas do IMDb.

Entrada : pasta dados_brutos/ com os 5 arquivos TSV (title_basics, title_ratings,
          title_crew, title_principals, name_basics), descompactados.
Saída   : dados/base_integrada.csv (um filme por linha, 15 colunas)

Regras (seções 4.3 e 4.4 do artigo):
  1. ratings: nota e votos numéricos, sem ausências, numVotes >= 100;
     fica com os 20.000 títulos mais votados.
  2. basics: junção interna; titleType = movie e isAdult = 0.
  3. \\N = ausente; ano e duração com to_numeric(errors="coerce");
     exclui quem não tem ano, duração, gêneros, nota ou votos;
     duração entre 50 e 300 minutos (inclusive); sem duplicidade de tconst.
  4. direção e elenco por junção à esquerda; elenco = até 3 ocorrências de
     actor/actress, por ordering; ausência = lista vazia.
"""
from pathlib import Path
import csv

import pandas as pd

RAIZ = Path("/content/projeto")
BRUTOS = RAIZ / "dados_brutos"
SAIDA = RAIZ / "dados"
SAIDA.mkdir(exist_ok=True)

TAM_BLOCO = 1_000_000
LEITURA = dict(sep="\t", na_values="\\N", dtype=str, quoting=csv.QUOTE_NONE)


def achar(nome):
    """Localiza o arquivo pelo nome (aceita title.basics.tsv, title_basics.tsv, .gz...)."""
    for p in sorted(BRUTOS.rglob("*")):
        n = p.name.lower().replace(".", "_")
        if p.is_file() and p.suffix.lower() != ".zip" and n.startswith(nome):
            return p
    raise FileNotFoundError(
        f"Não encontrei '{nome}' em {BRUTOS}. Coloque os 5 arquivos TSV nessa pasta "
        "(descompacte o ZIP do Kaggle)."
    )


def ler_filtrando(nome, coluna, valores, colunas=None):
    """Lê o arquivo em blocos e guarda só as linhas cujo 'coluna' está em 'valores'."""
    partes = []
    for bloco in pd.read_csv(achar(nome), chunksize=TAM_BLOCO, usecols=colunas, **LEITURA):
        partes.append(bloco[bloco[coluna].isin(valores)])
    return pd.concat(partes, ignore_index=True)


# 1. Avaliações
print("1/5 Lendo title_ratings...")
rat = pd.read_csv(achar("title_ratings"), **LEITURA)
rat["averageRating"] = pd.to_numeric(rat["averageRating"], errors="coerce")
rat["numVotes"] = pd.to_numeric(rat["numVotes"], errors="coerce")
rat = rat.dropna(subset=["averageRating", "numVotes"])
rat = rat[rat["numVotes"] >= 100]
rat = rat.nlargest(20000, "numVotes")
ids = set(rat["tconst"])
print(f"    títulos pré-selecionados: {len(rat)}")

# 2. Informações básicas + junção interna
print("2/5 Lendo title_basics...")
bas = ler_filtrando("title_basics", "tconst", ids)
df = bas.merge(rat, on="tconst", how="inner")
df["isAdult"] = pd.to_numeric(df["isAdult"], errors="coerce")
df = df[(df["titleType"] == "movie") & (df["isAdult"] == 0)]

# 3. Limpeza
df["startYear"] = pd.to_numeric(df["startYear"], errors="coerce")
df["runtimeMinutes"] = pd.to_numeric(df["runtimeMinutes"], errors="coerce")
df = df.dropna(subset=["startYear", "runtimeMinutes", "genres", "averageRating", "numVotes"])
df = df[df["runtimeMinutes"].between(50, 300)]
df = df.drop_duplicates("tconst").reset_index(drop=True)
df["startYear"] = df["startYear"].astype(int)
df["runtimeMinutes"] = df["runtimeMinutes"].astype(int)
df["isAdult"] = df["isAdult"].astype(int)
ids = set(df["tconst"])
print(f"    filmes após filtros: {len(df)}")

# 4. Direção
print("3/5 Lendo title_crew (direção)...")
crew = ler_filtrando("title_crew", "tconst", ids, colunas=["tconst", "directors"])
dir_map = {
    t: [x for x in (d.split(",") if isinstance(d, str) else []) if x]
    for t, d in zip(crew["tconst"], crew["directors"])
}
df["diretores_ids"] = df["tconst"].map(lambda t: dir_map.get(t, []))

# 5. Elenco (actor/actress, até 3 por filme, ordem do arquivo)
print("4/5 Lendo title_principals (elenco) - pode demorar alguns minutos...")
pr = ler_filtrando("title_principals", "tconst", ids,
                   colunas=["tconst", "ordering", "nconst", "category"])
pr = pr[pr["category"].isin(["actor", "actress"])].copy()
pr["ordering"] = pd.to_numeric(pr["ordering"], errors="coerce")
pr = pr.sort_values(["tconst", "ordering"]).groupby("tconst").head(3)
elenco_map = pr.groupby("tconst")["nconst"].apply(list).to_dict()
df["elenco_ids"] = df["tconst"].map(lambda t: elenco_map.get(t, []))

# 6. Nomes (só para leitura/explicação; o vetor usa os identificadores)
print("5/5 Lendo name_basics (nomes)...")
precisa = {n for l in df["diretores_ids"] for n in l} | {n for l in df["elenco_ids"] for n in l}
nomes = ler_filtrando("name_basics", "nconst", precisa, colunas=["nconst", "primaryName"])
nome_de = dict(zip(nomes["nconst"], nomes["primaryName"]))
df["diretores_nomes"] = df["diretores_ids"].map(lambda l: [nome_de.get(n, n) for n in l])
df["elenco_nomes"] = df["elenco_ids"].map(lambda l: [nome_de.get(n, n) for n in l])

colunas = ["tconst", "titleType", "primaryTitle", "originalTitle", "isAdult", "startYear",
           "endYear", "runtimeMinutes", "genres", "averageRating", "numVotes",
           "diretores_ids", "elenco_ids", "diretores_nomes", "elenco_nomes"]
df = df[colunas]
df.to_csv(SAIDA / "base_integrada.csv", index=False)

# Conferência com o artigo (valores esperados entre parênteses)
generos = df["genres"].str.split(",").explode().nunique()
print("\n=== CONFERÊNCIA COM O ARTIGO ===")
print(f"Filmes ................ {len(df)}   (14375)")
print(f"Colunas ............... {df.shape[1]}   (15)")
print(f"Gêneros distintos ..... {generos}   (24)")
print(f"Anos .................. {df.startYear.min()} a {df.startYear.max()}   (1915 a 2022)")
print(f"Menor nº de votos ..... {df.numVotes.min():.0f}   (5288)")
print(f"Sem diretores ......... {(df.diretores_ids.map(len) == 0).sum()}   (3)")
print(f"Sem elenco ............ {(df.elenco_ids.map(len) == 0).sum()}   (336)")
print(f"\nSalvo em: {SAIDA / 'base_integrada.csv'}")

## 5. Etapa 2 - Modelos A e B e métricas
Compare com a Tabela 5 do artigo.

In [ ]:
"""
Etapa 2 - Modelagem (Modelos A e B) e avaliação Top 5 (seções 4.5 e 4.6 do artigo).

Entrada : dados/base_integrada.csv
Saída   : resultados/metricas.csv e resultados/recomendacoes.csv
"""
from pathlib import Path
import ast

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction import DictVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

RAIZ = Path("/content/projeto")
SAIDA = RAIZ / "resultados"
SAIDA.mkdir(exist_ok=True)

PESO_GENEROS, PESO_NUMERICOS, PESO_EQUIPE = 2.0, 0.35, 0.8
SEMENTE, N_CONSULTAS, TOP = 42, 100, 5

df = pd.read_csv(RAIZ / "dados" / "base_integrada.csv")
for c in ["diretores_ids", "elenco_ids"]:
    df[c] = df[c].apply(ast.literal_eval)
df["generos"] = df["genres"].str.split(",")

# --- Representação vetorial ---
G = MultiLabelBinarizer(sparse_output=True).fit_transform(df["generos"])
numericos = np.column_stack([df["startYear"], df["runtimeMinutes"],
                             df["averageRating"], np.log1p(df["numVotes"])])
Z = csr_matrix(StandardScaler().fit_transform(numericos))
registros = [
    {**{f"diretor_{d}": 1 for d in dirs}, **{f"ator_{a}": 1 for a in elenco}}
    for dirs, elenco in zip(df["diretores_ids"], df["elenco_ids"])
]
E = DictVectorizer().fit_transform(registros)

X_A = hstack([PESO_GENEROS * G, PESO_NUMERICOS * Z], format="csr")
X_B = hstack([PESO_GENEROS * G, PESO_NUMERICOS * Z, PESO_EQUIPE * E], format="csr")
print(f"Modelo A: {X_A.shape[1]} dimensões, {X_A.nnz} não nulos (artigo: 28 e 93.692)")
print(f"Modelo B: {X_B.shape[1]} dimensões, {X_B.nnz} não nulos (artigo: 22.328 e 151.606)")

# --- Consultas ---
rng = np.random.default_rng(SEMENTE)
consultas = rng.choice(len(df), size=N_CONSULTAS, replace=False)


def recomendar(X):
    nn = NearestNeighbors(n_neighbors=TOP + 1, metric="cosine", algorithm="brute").fit(X)
    dist, ind = nn.kneighbors(X[consultas])
    saida = []
    for q, linha_d, linha_i in zip(consultas, dist, ind):
        pares = [(int(j), 1 - float(d)) for j, d in zip(linha_i, linha_d) if j != q]
        saida.append(pares[:TOP])
    return saida


generos = [set(g) for g in df["generos"]]
equipe = [set(d) | set(e) for d, e in zip(df["diretores_ids"], df["elenco_ids"])]


def metricas(resultado):
    cg, se, dg = [], [], []
    for q, pares in zip(consultas, resultado):
        viz = [j for j, _ in pares]
        cg.append(sum(bool(generos[q] & generos[j]) for j in viz) / TOP)
        se.append(sum(bool(equipe[q] & equipe[j]) for j in viz) / TOP)
        dg.append(len(set().union(*(generos[j] for j in viz))))
    return float(np.mean(cg)), float(np.mean(se)), float(np.mean(dg))


res = {"A": recomendar(X_A), "B": recomendar(X_B)}

linhas_m, linhas_r = [], []
for modelo, resultado in res.items():
    cg, se, dg = metricas(resultado)
    linhas_m.append({"Modelo": modelo, "Precisao_genero_5": cg,
                     "Sobreposicao_equipe_5": se, "Diversidade_media": dg})
    for q, pares in list(zip(consultas, resultado))[:5]:
        for pos, (j, sim) in enumerate(pares, start=1):
            linhas_r.append({"Modelo": modelo, "Consulta": df.loc[q, "primaryTitle"],
                             "Posicao": pos, "Recomendacao": df.loc[j, "primaryTitle"],
                             "Similaridade": round(sim, 4)})

pd.DataFrame(linhas_m).to_csv(SAIDA / "metricas.csv", index=False)
pd.DataFrame(linhas_r).to_csv(SAIDA / "recomendacoes.csv", index=False)

print("\n=== CONFERÊNCIA COM O ARTIGO (Tabela 5) ===")
for m in linhas_m:
    print(f"Modelo {m['Modelo']}: CG@5={m['Precisao_genero_5']:.3f}  "
          f"SE@5={m['Sobreposicao_equipe_5']*100:.1f}%  DG@5={m['Diversidade_media']:.2f}")
print("Artigo : A -> CG 1.000, SE 4,0%, DG 2,61 | B -> CG 1.000, SE 24,6%, DG 2,60")
print(f"\nSalvo em: {SAIDA}")

## 6. Etapa 3 - Gráficos

In [ ]:
"""
Etapa 3 - Gráficos (Figuras 1 a 5 do artigo).

Entrada : dados/base_integrada.csv e resultados/metricas.csv
Saída   : resultados/graficos/*.png
"""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # só salva os arquivos, sem abrir janelas
import matplotlib.pyplot as plt
import seaborn as sns

RAIZ = Path("/content/projeto")
SAIDA = RAIZ / "resultados" / "graficos"
SAIDA.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAIZ / "dados" / "base_integrada.csv")
df["generos"] = df["genres"].str.split(",")
sns.set_theme(style="whitegrid")

# Figura 1 - gêneros mais frequentes
freq = df.explode("generos").generos.value_counts().head(12)
plt.figure(figsize=(10, 5))
sns.barplot(x=freq.values, y=freq.index)
plt.title("Generos mais frequentes")
plt.xlabel("Quantidade de filmes")
plt.tight_layout()
plt.savefig(SAIDA / "grafico_generos.png", dpi=160)
plt.close()

# Figura 2 - nota média por década
df["decada"] = (df.startYear // 10 * 10).astype(int)
decadas = df.groupby("decada").averageRating.mean()
plt.figure(figsize=(10, 5))
sns.lineplot(x=decadas.index, y=decadas.values, marker="o")
plt.title("Nota media por decada")
plt.ylabel("Nota media")
plt.tight_layout()
plt.savefig(SAIDA / "grafico_decadas.png", dpi=160)
plt.close()

# Figura 3 - duração x avaliação
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="runtimeMinutes", y="averageRating", alpha=0.25, s=15)
plt.title("Duracao x avaliacao")
plt.tight_layout()
plt.savefig(SAIDA / "grafico_duracao.png", dpi=160)
plt.close()

# Figura 4 - popularidade x avaliação
plt.figure(figsize=(9, 5))
sns.scatterplot(x=np.log1p(df.numVotes), y=df.averageRating, alpha=0.25, s=15)
plt.title("Popularidade x avaliacao")
plt.xlabel("log(1 + numero de votos)")
plt.tight_layout()
plt.savefig(SAIDA / "grafico_votos.png", dpi=160)
plt.close()

# Figura 5 - comparativo dos modelos
metricas = pd.read_csv(RAIZ / "resultados" / "metricas.csv")
percentuais = metricas["Sobreposicao_equipe_5"] * 100
rotulos = "Modelo " + metricas["Modelo"].astype(str)
fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(rotulos, percentuais)
ax.set_title("Sobreposição de profissionais nas recomendações")
ax.set_xlabel("Modelo")
ax.set_ylabel("Recomendações com profissional em comum (%)")
ax.set_ylim(0, max(30, float(percentuais.max()) * 1.2))
ax.bar_label(barras, labels=[f"{v:.1f}%".replace(".", ",") for v in percentuais], padding=4)
fig.tight_layout()
fig.savefig(SAIDA / "grafico_comparacao_modelos.png", dpi=160)
plt.close(fig)

print("Gráficos salvos em:", SAIDA)
for p in sorted(SAIDA.glob("*.png")):
    print(" -", p.name)

In [ ]:
from IPython.display import Image, display
for p in sorted((RAIZ / "resultados" / "graficos").glob("*.png")):
    print(p.name)
    display(Image(filename=str(p), width=700))

## 6b. Etapa 4 - Conferência com as tabelas do artigo
Cada linha mostra **OK** ou **DIFERE**. O relatório também é salvo em `resultados/conferencia.txt`.

In [ ]:
"""
Etapa 4 - Conferência: recalcula as tabelas do artigo (2 a 6 e Apêndice A) a partir dos
arquivos gerados e compara com os valores publicados.

Entrada : dados/base_integrada.csv, resultados/metricas.csv, resultados/recomendacoes.csv
Saída   : relatório no terminal (OK / DIFERE) e resultados/conferencia.txt
"""
from pathlib import Path
import ast
import io
import re
import unicodedata
from contextlib import redirect_stdout

import numpy as np
import pandas as pd

RAIZ = Path("/content/projeto")
placar = {"ok": 0, "difere": 0}


def checar(nome, obtido, esperado, tol=0.0):
    try:
        bate = abs(float(obtido) - float(esperado)) <= tol
    except (TypeError, ValueError):
        bate = str(obtido) == str(esperado)
    placar["ok" if bate else "difere"] += 1
    print(f"[{'OK    ' if bate else 'DIFERE'}] {nome}: obtido={obtido}  esperado={esperado}")


def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", " ", s).strip()


def conferir():
    df = pd.read_csv(RAIZ / "dados" / "base_integrada.csv")
    for c in ["diretores_ids", "elenco_ids"]:
        if c in df.columns:
            df[c] = df[c].apply(ast.literal_eval)
    df["generos"] = df["genres"].str.split(",")

    print("=== Seção 4.3 / 5.1 - Base final ===")
    checar("Filmes", len(df), 14375)
    checar("Colunas", df.shape[1], 15)
    checar("Gêneros distintos", df.generos.explode().nunique(), 24)
    checar("tconst únicos", df.tconst.nunique(), 14375)
    checar("Todos titleType = movie", bool((df.titleType == "movie").all()), True)
    checar("Todos isAdult = 0", bool((df.isAdult == 0).all()), True)

    print("\n=== Tabela 2 - Estatísticas descritivas ===")
    for nome, col, esp in [
        ("Ano", "startYear", dict(min=1915, median=2006, mean=2000.12, max=2022)),
        ("Duração", "runtimeMinutes", dict(min=50, median=104, mean=108.56, max=288)),
        ("Nota", "averageRating", dict(min=1.0, median=6.6, mean=6.55, max=9.7)),
        ("Votos", "numVotes", dict(min=5288, median=17382, mean=61609.51, max=2578898)),
    ]:
        for est, valor in esp.items():
            tol = 0.005 if est == "mean" else 0.0
            checar(f"{nome} - {est}", round(getattr(df[col], est)(), 2), valor, tol)
    checar("Duração - 1º quartil", df.runtimeMinutes.quantile(.25), 94)
    checar("Duração - 3º quartil", df.runtimeMinutes.quantile(.75), 118)

    print("\n=== Seção 5.1 - Direção e elenco ===")
    n_elenco = df.elenco_ids.map(len)
    checar("Elenco com 3", int((n_elenco == 3).sum()), 13958)
    checar("Elenco com 1", int((n_elenco == 1).sum()), 42)
    checar("Elenco com 2", int((n_elenco == 2).sum()), 39)
    checar("Elenco vazio", int((n_elenco == 0).sum()), 336)
    checar("Sem diretores", int((df.diretores_ids.map(len) == 0).sum()), 3)

    print("\n=== Tabela 3 - Gêneros ===")
    longa = df.explode("generos")
    esperado3 = {"Drama": (8094, 6.81), "Comedy": (5132, 6.38), "Action": (3242, 6.31),
                 "Crime": (2862, 6.61), "Romance": (2495, 6.61), "Thriller": (2345, 6.29),
                 "Adventure": (2135, 6.41), "Horror": (1782, 5.86), "Mystery": (1373, 6.38),
                 "Fantasy": (991, 6.24), "Biography": (960, 7.07), "Sci-Fi": (939, 6.10)}
    freq = longa.generos.value_counts()
    media = longa.groupby("generos").averageRating.mean()
    for g, (n, m) in esperado3.items():
        checar(f"{g} - filmes", int(freq.get(g, 0)), n)
        checar(f"{g} - nota média", round(float(media.get(g, np.nan)), 2), m, 0.005)
    checar("Top 12 na ordem do artigo", list(freq.head(12).index), list(esperado3))

    print("\n=== Tabela 4 - Décadas ===")
    df["decada"] = (df.startYear // 10 * 10).astype(int)
    por_dec = df.groupby("decada").averageRating.agg(["size", "mean"])
    esperado4 = {1910: (3, 7.07), 1920: (50, 7.69), 1930: (153, 7.42), 1940: (242, 7.43),
                 1950: (356, 7.36), 1960: (496, 7.22), 1970: (674, 7.05), 1980: (1187, 6.59),
                 1990: (1944, 6.52), 2000: (3487, 6.43), 2010: (4953, 6.40), 2020: (830, 6.29)}
    for d, (n, m) in esperado4.items():
        n_obt = int(por_dec["size"].get(d, 0))
        m_obt = round(float(por_dec["mean"].get(d, np.nan)), 2)
        checar(f"{d} - filmes", n_obt, n)
        checar(f"{d} - nota média", m_obt, m, 0.005)

    print("\n=== Seções 5.4 e 5.5 - Correlações ===")
    lv = np.log1p(df.numVotes)
    checar("Pearson duração x nota", round(df.runtimeMinutes.corr(df.averageRating), 3), 0.309, 0.0005)
    checar("Spearman duração x nota",
           round(df.runtimeMinutes.corr(df.averageRating, method="spearman"), 3), 0.331, 0.0005)
    checar("Pearson ln(1+votos) x nota", round(lv.corr(df.averageRating), 3), 0.214, 0.0005)
    checar("Spearman votos x nota",
           round(df.numVotes.corr(df.averageRating, method="spearman"), 3), 0.171, 0.0005)

    print("\n=== Tabela 5 - Métricas dos modelos ===")
    met = pd.read_csv(RAIZ / "resultados" / "metricas.csv").set_index("Modelo")
    for m, (cg, se, dg) in {"A": (1.0, 0.040, 2.61), "B": (1.0, 0.246, 2.60)}.items():
        checar(f"Modelo {m} - CG@5", round(met.loc[m, "Precisao_genero_5"], 3), cg, 0.0005)
        checar(f"Modelo {m} - SE@5", round(met.loc[m, "Sobreposicao_equipe_5"], 3), se, 0.0005)
        checar(f"Modelo {m} - DG@5", round(met.loc[m, "Diversidade_media"], 2), dg, 0.005)
    checar("Pares com equipe em comum - A (de 500)", round(met.loc["A", "Sobreposicao_equipe_5"] * 500), 20)
    checar("Pares com equipe em comum - B (de 500)", round(met.loc["B", "Sobreposicao_equipe_5"] * 500), 123)

    print("\n=== Tabela 6 e Apêndice A - Recomendações ===")
    esperado_a = {
        "connie": {
            "A": [("Kill Your Friends", .9979), ("Sound of Noise", .9945), ("Diva", .9868),
                  ("The Adventure of Sherlock Holmes' Smarter Brother", .9855), ("Airheads", .9848)],
            "B": [("Kill Your Friends", .8242), ("The Adventure of Sherlock Holmes' Smarter Brother", .8154),
                  ("Diva", .8152), ("Airheads", .8132), ("Be Cool", .8114)]},
        "mcqueen": {
            "A": [("Val", .9993), ("De Palma", .9985), ("Listen to Me Marlon", .9978),
                  ("I Am Greta", .9968), ("Life Itself", .9959)],
            "B": [("I Am Greta", .8948), ("Life Itself", .8935),
                  ("Steve Jobs: The Man in the Machine", .8889), ("Won't You Be My Neighbor?", .8866),
                  ("The Wolfpack", .8865)]},
        "altitude": {
            "A": [("Await Further Instructions", .9990), ("Banshee Chapter", .9982), ("Eden Log", .9979),
                  ("Area 51", .9978), ("What Lies Below", .9975)],
            "B": [("What Lies Below", .8303), ("Area 51", .8299), ("Await Further Instructions", .8297),
                  ("Phoenix Forgotten", .8281), ("Banshee Chapter", .8280)]},
        "cach": {
            "A": [("We Need to Talk About Kevin", .9977), ("First Reformed", .9973), ("The Gift", .9960),
                  ("Copycat", .9957), ("Martha Marcy May Marlene", .9954)],
            "B": [("The White Ribbon", .8644), ("We Need to Talk About Kevin", .8260),
                  ("Tinker Tailor Soldier Spy", .8246), ("The Gift", .8243), ("First Reformed", .8243)]},
        "okkadu": {
            "A": [("Heropanti", .7875), ("Lakshya", .7006), ("Arjun Reddy", .6985),
                  ("Sarpatta Parambarai", .6970), ("I", .6970)],
            "B": [("Heropanti", .6342), ("Ghilli", .6221), ("In Your Name", .5975),
                  ("Lakshya", .5894), ("Arjun Reddy", .5875)]},
    }
    rec = pd.read_csv(RAIZ / "resultados" / "recomendacoes.csv")

    def coluna(padroes, excluir=()):
        for c in rec.columns:
            if c not in excluir and any(norm(c).startswith(p) or p in norm(c) for p in padroes):
                return c
        return None

    c_mod = coluna(["model"])
    c_con = coluna(["consulta", "entrada", "query", "filme"], excluir=[c_mod])
    c_rec = coluna(["recomend", "vizinho", "sugest"], excluir=[c_mod, c_con])
    c_sim = coluna(["sim", "score", "escore"], excluir=[c_mod, c_con, c_rec])
    c_pos = coluna(["posicao", "pos", "rank", "ordem"], excluir=[c_mod, c_con, c_rec, c_sim])
    if None in (c_mod, c_con, c_rec, c_sim):
        print("[DIFERE] Não consegui identificar as colunas de recomendacoes.csv:", list(rec.columns))
        placar["difere"] += 1
        return
    rec["_modelo"] = rec[c_mod].astype(str).str.strip().str[-1].str.upper()
    rec["_consulta"] = rec[c_con].map(norm)
    rec["_pos"] = rec[c_pos] if c_pos else rec.groupby(["_modelo", "_consulta"]).cumcount() + 1

    for chave, por_modelo in esperado_a.items():
        for modelo, lista in por_modelo.items():
            sub = rec[(rec._modelo == modelo) & rec._consulta.str.contains(chave)].sort_values("_pos")
            if sub.empty:
                print(f"[DIFERE] Consulta '{chave}' (modelo {modelo}) não está em recomendacoes.csv")
                placar["difere"] += 1
                continue
            for i, (titulo, sim) in enumerate(lista):
                if i >= len(sub):
                    print(f"[DIFERE] {chave}/{modelo}/#{i+1}: faltando")
                    placar["difere"] += 1
                    continue
                linha = sub.iloc[i]
                checar(f"{chave}/{modelo}/#{i+1} título", norm(linha[c_rec]), norm(titulo))
                checar(f"{chave}/{modelo}/#{i+1} similaridade", round(float(linha[c_sim]), 4), sim, 0.00006)


buf = io.StringIO()
with redirect_stdout(buf):
    conferir()
    print(f"\nRESUMO: {placar['ok']} valores conferem, {placar['difere']} diferem.")
texto = buf.getvalue()
print(texto)
(RAIZ / "resultados").mkdir(exist_ok=True)
(RAIZ / "resultados" / "conferencia.txt").write_text(texto, encoding="utf-8")

## 7. Baixar os resultados (para enviar ao GitHub)

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/resultados_projeto", "zip", RAIZ, "resultados")
zip_dados = shutil.make_archive("/content/dados_projeto", "zip", RAIZ, "dados")
files.download(zip_path)
files.download(zip_dados)